In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd
import sys
from scipy.stats import spearmanr
import subprocess

## **Functions**

In [ ]:
base_dir = os.getcwd()
results_dir = os.path.join(base_dir, "results_trRosetta")
trRosetta_dir = os.path.join(base_dir, "trRosetta")
results_fasta_dir = os.path.join(results_dir, "fasta_files")
results_msa_dir = os.path.join(results_dir, "msa")
results_matrices_dir = os.path.join(results_dir, "matrix_distances")
real_distances_dir = os.path.join(base_dir, "real_distances")

Function to create a FASTA file from a sequence

In [ ]:
def create_fasta_from_sequence(sequence, sequence_id):

    base_dir = os.getcwd()
    os.makedirs(results_fasta_dir, exist_ok=True)
    fasta_path = os.path.join(results_fasta_dir, f"{sequence_id}.fasta")
    
    # Write FASTA format
    with open(fasta_path, 'w') as f:
        f.write(f">{sequence_id}\n")
        f.write(f"{sequence}\n")
        
    print(f"FASTA file successfully created at:\n{fasta_path}")
    return fasta_path

Function to create a MSA from a fasta file

In [ ]:
if trRosetta_dir not in sys.path:
    sys.path.append(trRosetta_dir)

# Import MSA generation function
try:
    from generate_msa_with_colabfold import generate_msa_and_clean
except ImportError:
    print("Error: Unable to import 'generate_msa_with_colabfold.py'.")
    print(f"Verify that the file exists in: {trRosetta_dir}")


def sanitize_a3m(file_path):
    """
    Clean an A3M file by removing insertions and enforcing uniform sequence length.
    """
    if not os.path.exists(file_path):
        return
    
    with open(file_path, 'r') as f:
        lines = f.readlines()
        
    cleaned_lines = []
    reference_length = 0
    
    for line in lines:
        line = line.strip()
        
        if not line or line.startswith("#"):
            continue 
            
        if line.startswith(">"):
            cleaned_lines.append(line)
        else:
            seq = ''.join(c for c in line if not c.islower())
            
            if reference_length == 0:
                reference_length = len(seq)
                
            if len(seq) == reference_length:
                cleaned_lines.append(seq)
                
    with open(file_path, 'w') as f:
        f.write('\n'.join(cleaned_lines) + '\n')


def get_msa(fasta_path, sequence_id):
    """
    Generate or reuse an MSA and return the sanitized A3M path.
    """
    
    msa_dir = os.path.join(results_msa_dir, sequence_id)
    a3m_file = os.path.join(msa_dir, f"{sequence_id}.a3m")
    
    if os.path.exists(a3m_file):
        print(f"\nExisting alignment detected for {sequence_id}. Skipping computation.")
    else:
        print(f"\nGenerating MSA for {sequence_id}...")
        generate_msa_and_clean(fasta_path, results_msa_dir)
        
    if os.path.exists(a3m_file):
        sanitize_a3m(a3m_file)
        print(f"A3M file ready: {a3m_file}")
        return a3m_file
    else:
        raise FileNotFoundError(f"Expected file not found: {a3m_file}")

Function to predict distance matrix with trRosetta using msa

In [ ]:
# Python interpreter corresponding to the trRosetta environment
PYTHON_TRROSETTA = "/home/biocomp/anaconda3/envs/trrosetta_env/bin/python"
SCRIPT_EXTRAER = os.path.join(trRosetta_dir, "distance_matrix.trRosetta.py")

def predict_distance_matrix(a3m_path, sequence_id):
    """
    Run the trRosetta neural network, export the distance matrix to CSV,
    and remove the intermediate .npz file.
    """
    if not os.path.exists(results_matrices_dir):
        os.makedirs(results_matrices_dir)
        
    npz_file = os.path.join(results_matrices_dir, f"{sequence_id}_prediction.npz")
    csv_file = os.path.join(results_matrices_dir, f"{sequence_id}_distances.csv")
    
    if os.path.exists(csv_file):
        print(f"\nExisting distance matrix detected for {sequence_id}.")
        print(f"Using stored matrix at:\n{csv_file}")
        return csv_file

    print(f"\nStarting neural network prediction for {sequence_id}...")
    print("Suppressing TensorFlow warnings; this may take some time.")
    
    silent_env = os.environ.copy()
    silent_env["TF_CPP_MIN_LOG_LEVEL"] = "3"
    silent_env["PYTHONWARNINGS"] = "ignore"
    
    # GPU
    silent_env["CUDA_VISIBLE_DEVICES"] = "0"
    current_dir = os.getcwd()
    
    try:
        os.chdir(trRosetta_dir)
        
        # 1. Run prediction (generates the .npz file)
        predict_cmd = [
            PYTHON_TRROSETTA,
            "network/predict.py",
            "-m", "model2019_07",
            a3m_path,
            npz_file
        ]
        subprocess.run(predict_cmd, check=True, env=silent_env)
        
        # 2. Convert to CSV
        print("Extracting distance matrix to CSV format...")
        extract_cmd = [PYTHON_TRROSETTA, SCRIPT_EXTRAER, npz_file, csv_file]
        subprocess.run(extract_cmd, check=True, env=silent_env)
        
        # 3. Remove intermediate .npz file
        if os.path.exists(npz_file):
            os.remove(npz_file)
            print("Intermediate .npz file removed to save space.")
        
        print(f"Distance matrix successfully saved at:\n{csv_file}")
        return csv_file
        
    except subprocess.CalledProcessError as e:
        print("\nError during trRosetta execution: process failed.")
        print(f"Details: {e}")
        raise
    finally:
        os.chdir(current_dir)

Fitness calculation function

In [ ]:
def get_fitness(predicted_csv, target_name):
    """
    Computes a unified fitness score by combining Distance-Weighted Error (DWE)
    and Spearman correlation under a conditional scheme.
    """
    real_csv = os.path.join(real_distances_dir, f"{target_name}.csv")
    
    if not os.path.exists(real_csv):
        raise FileNotFoundError(
            f"Real distance matrix for target '{target_name}' not found at:\n{real_csv}"
        )
        
    df_real = pd.read_csv(real_csv)
    df_pred = pd.read_csv(predicted_csv)
    
    mat_real = df_real.values
    mat_pred = df_pred.values
    
    if mat_real.shape != mat_pred.shape:
        raise ValueError(
            f"Dimension mismatch: real {mat_real.shape} vs predicted {mat_pred.shape}."
        )
        
    upper_indices = np.triu_indices(mat_real.shape[0], k=1)
    dist_real = mat_real[upper_indices]
    dist_pred = mat_pred[upper_indices]
    
    epsilon = 1e-8
    dwe = np.mean(np.abs(dist_real - dist_pred) / (dist_real + epsilon))
    
    corr, _ = spearmanr(dist_real, dist_pred)
    
    if corr == -1:
        fitness = np.inf
    elif corr < 0:
        fitness = (1 / (2 * corr)) * dwe
    elif corr == 0:
        fitness = dwe
    else:
        fitness = (1 / corr) * dwe
'''
    print("\n*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*")
    print(f" Unified Fitness for: {target_name}")
    print("*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*")
    print(f"DWE:        {dwe:.4f}")
    print(f"Spearman:   {corr:.4f}")
    print(f"Fitness:    {fitness:.4f}")
    print("*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*")
    
    return fitness
'''

## **Genetic algorithm**

In [ ]:
base_dir = os.getcwd()
genetic_functions_dir = os.path.join(base_dir, "genetic")
initial_pop_dir = os.path.join(base_dir, "initial_population")
import random

In [ ]:
def tournament(population, k=2):
    contenders = random.sample(population, k)
    winner = min(contenders, key=lambda individual: individual['fitness'])
    return winner

In [ ]:
POLAR_AA = ['D', 'E', 'R', 'K', 'H', 'N', 'Q', 'S', 'T', 'Y']
NONPOLAR_AA = ['A', 'G', 'V', 'L', 'I', 'P', 'F', 'M', 'W', 'C']

def mutation(sequence, mutation_prob=0.01):
    new_sequence = []
    for aa in sequence:
        if random.random() < mutation_prob:
            if aa in POLAR_AA:
                options = [option for option in POLAR_AA if option != aa]
                new_aa = random.choice(options)
            elif aa in NONPOLAR_AA:
                options = [option for option in NONPOLAR_AA if option != aa]
                new_aa = random.choice(options)
            else:
                new_aa = aa
            new_sequence.append(new_aa)
        else:
            new_sequence.append(aa)
    return "".join(new_sequence)

In [ ]:
def crossover(sequence1, sequence2):
    if len(sequence1) != len(sequence2):
        raise ValueError("Sequences must have the same length.")
    length = len(sequence1)
    point1 = random.randint(0, length - 2)
    point2 = random.randint(point1 + 1, length)
    child1 = sequence1[:point1] + sequence2[point1:point2] + sequence1[point2:]
    child2 = sequence2[:point1] + sequence1[point1:point2] + sequence2[point2:]
    return child1, child2

In [ ]:
import concurrent.futures

def evaluate_batch_parallel(sequences, target_protein, max_workers=2):
    """
    Evaluates a list of sequences in parallel.
    max_workers: Number of trRosetta processes to run on the GPU simultaneously.
    """
    results = []
    # ThreadPoolExecutor manages the queue of worker threads
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        # Prepare all the "jobs"
        futures = {
            executor.submit(evaluate_sequence, seq, target_protein): seq
            for seq in sequences
        }
        
        # As they complete (regardless of order), store the results
        for future in concurrent.futures.as_completed(futures):
            seq = futures[future]
            try:
                fit = future.result()
                results.append({'sequence': seq, 'fitness': fit})
            except Exception as exc:
                print(f"Parallel error with sequence {seq[:10]}... : {exc}")
                results.append({'sequence': seq, 'fitness': float('inf')})
                
    return results

In [ ]:
# --- FITNESS CACHE ---
# This dictionary prevents evaluating the same sequence more than once
fitness_cache = {}

def evaluate_sequence(sequence, target_protein):
    """Evaluates a sequence using the trRosetta pipeline. Uses caching for efficiency."""
    # 1. Check if we already know the result
    if sequence in fitness_cache:
        return fitness_cache[sequence]
    
    # 2. If it's new, run it through your pipeline
    try:
        fasta_path = create_fasta_from_sequence(sequence, sequence)  # ID = sequence
        msa_path = get_msa(fasta_path, sequence)
        matrix_path = predict_distance_matrix(msa_path, sequence)
        fitness = get_fitness(matrix_path, target_protein)
        
        # Store in cache
        fitness_cache[sequence] = fitness
        return fitness
        
    except Exception as e:
        print(f"Error processing sequence {sequence[:10]}... : {e}")
        # Return a very high (bad) fitness so the algorithm discards it
        return float('inf')

In [ ]:
def genetic_algorithm_trRosetta(target_protein, generations=300, children_count=50, p_best=200, max_gpu_workers=2):
    print(f"Starting GA for protein: {target_protein} with {children_count} children per generation.")
    
    random.seed(26)
    # 1. Load the initial population
    csv_path = os.path.join(initial_pop_dir, f"{target_protein}.csv")
    df = pd.read_csv(csv_path)
    initial_sequences = df['Sequence'].tolist()
    
    # Evaluate initial population (can also be done in parallel)
    print("Evaluating initial population in parallel batches...")
    population = evaluate_batch_parallel(initial_sequences, target_protein, max_workers=max_gpu_workers)
    population = sorted(population, key=lambda x: x['fitness'])[:p_best]
    
    # 2. Generational loop
    for gen in range(generations):
        current_k = 2 if gen < 200 else 3
        print(f"\n--- Generation {gen + 1}/{generations} | Tournament k={current_k} ---")
        
        children_sequences_raw = []
        
        # Generate offspring sequences (crossover and mutation on CPU, fast)
        for _ in range(children_count // 2):
            parent1 = tournament(population, k=current_k)
            parent2 = tournament(population, k=current_k)
            child_seq1, child_seq2 = crossover(parent1['sequence'], parent2['sequence'])
            children_sequences_raw.extend([mutation(child_seq1), mutation(child_seq2)])
            
        # Filter to leverage cache efficiently
        unique_to_evaluate = [seq for seq in children_sequences_raw if seq not in fitness_cache]
        
        # Immediately retrieve known ones from cache
        children_population = [
            {'sequence': seq, 'fitness': fitness_cache[seq]}
            for seq in children_sequences_raw if seq in fitness_cache
        ]
        
        # Evaluate NEW sequences in parallel on GPU
        if unique_to_evaluate:
            print(f"Sending batch of {len(unique_to_evaluate)} new sequences to the GPU...")
            new_evaluations = evaluate_batch_parallel(
                unique_to_evaluate, target_protein, max_workers=max_gpu_workers
            )
            children_population.extend(new_evaluations)
            
        # (μ + λ) scheme
        combined_population = population + children_population
        combined_population = sorted(combined_population, key=lambda x: x['fitness'])
        population = combined_population[:p_best]
        
        print(f"Current best fitness: {population[0]['fitness']}")
        
    print("\nEvolution finished!")
    

    # 1. Convert the list of dictionaries from the final population into a DataFrame
    df_results = pd.DataFrame(population)

    # 2. Define the path and file name using the requested format
    file_name = f"genetic_results_{target_protein}.csv"
    save_path = os.path.join(results_dir, file_name)

    # 3. Save the DataFrame as CSV (index=False avoids saving the pandas index column)
    df_results.to_csv(save_path, index=False)

    print(f"Final population successfully saved to:\n{save_path}")

    # 4. Return the full population
    return population

In [ ]:
best_individual = genetic_algorithm_trRosetta(target_protein="1aay", generations=300, children_count=50, p_best=200)
print("The best individual was:", best_individual)